# Tango transcription with YourMT3 — `other.wav` only

This notebook:
1. Separates your audio with **demucs** on the Colab GPU.
2. Measures each stem and skips silent/empty ones.
3. Sends **only `other.wav`** (bandoneón/violin) to the **YourMT3** HuggingFace Space.
4. Downloads the resulting MIDI.

Processing only `other.wav` saves HuggingFace ZeroGPU quota and focuses on the stem YourMT3 handles best.

## Setup

1. Set runtime to **GPU**: `Runtime` → `Change runtime type` → `T4 GPU`.
2. (Optional but recommended) Get a HuggingFace token from https://huggingface.co/settings/tokens and paste it in the auth cell. Free accounts get more ZeroGPU quota than anonymous usage.

In [ ]:
# @title 1. Install dependencies
!pip install -q demucs gradio_client librosa soundfile

In [ ]:
# @title 2. Optional: authenticate with HuggingFace for more quota
# Paste your HuggingFace token here (https://huggingface.co/settings/tokens)
# Leave empty to use anonymous quota.
HF_TOKEN = ""  # @param {type:"string"}

import os
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("HF_TOKEN set.")
else:
    print("Using anonymous HuggingFace quota.")

In [ ]:
# @title 3. Upload your audio file
from google.colab import files

uploaded = files.upload()
audio_path = next(iter(uploaded))
print(f"Uploaded: {audio_path}")

In [ ]:
# @title 4. Run demucs source separation
import subprocess
from pathlib import Path

demucs_model = "htdemucs_ft"  # @param ["htdemucs", "htdemucs_ft", "htdemucs_6s"]

output_base = Path("/content/demucs_out")
output_base.mkdir(parents=True, exist_ok=True)

cmd = [
    "demucs",
    "--name", demucs_model,
    "--out", str(output_base),
    audio_path,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

song_name = Path(audio_path).stem
stems_dir = output_base / song_name / demucs_model / song_name
print("\nStems written to:", stems_dir)
for f in sorted(stems_dir.glob("*.wav")):
    print(" -", f.name)

In [ ]:
# @title 5. Detect silent/empty stems
import numpy as np
import soundfile as sf

def rms_db(path):
    """Return RMS loudness of a WAV file in dBFS."""
    x, sr = sf.read(str(path))
    if x.ndim > 1:
        x = x.mean(axis=1)
    rms = np.sqrt(np.mean(x**2))
    return 20 * np.log10(max(rms, 1e-10))

silence_threshold_db = -50  # @param {type:"number"}

stem_loudness = {}
for f in sorted(stems_dir.glob("*.wav")):
    db = rms_db(f)
    stem_loudness[f.stem] = db
    status = "silent" if db < silence_threshold_db else "active"
    print(f"{f.stem:12s} {db:6.1f} dBFS  -> {status}")

In [ ]:
# @title 6. Transcribe only `other.wav` with YourMT3
from gradio_client import Client, handle_file
import base64
import re

SPACE_ID = "mimbres/YourMT3"

def extract_midi_from_html(html):
    """Extract the first non-empty base64 MIDI from YourMT3 HTML output."""
    pattern = re.compile(r"data:audio/midi;base64,([^\s\"'<>]+)")
    matches = pattern.findall(html)
    if not matches:
        raise ValueError("No MIDI data URI found in response")
    for b64 in matches:
        data = base64.b64decode(b64)
        if len(data) > 50:
            return data
    return base64.b64decode(matches[0])

other_stem = stems_dir / "other.wav"
if not other_stem.exists():
    raise FileNotFoundError(f"other.wav not found at {other_stem}")

db = stem_loudness.get("other", rms_db(other_stem))
if db < silence_threshold_db:
    print("WARNING: other.wav appears silent. Skipping YourMT3 call.")
else:
    print(f"Transcribing other.wav ({db:.1f} dBFS) with YourMT3...")
    client = Client(SPACE_ID)
    result = client.predict(
        audio_filepath=handle_file(str(other_stem)),
        api_name="/process_audio",
    )
    midi_bytes = extract_midi_from_html(result)

    out_dir = Path("/content/yourmt3_out")
    out_dir.mkdir(parents=True, exist_ok=True)
    midi_path = out_dir / f"{song_name}_other_yourmt3.mid"
    midi_path.write_bytes(midi_bytes)
    print(f"\nSaved: {midi_path} ({len(midi_bytes)} bytes)")

    files.download(str(midi_path))

## Optional: transcribe additional stems

If you have quota left and want to try drums/bass/vocals, run the cell below. It will only send stems that are not silent.

In [ ]:
# @title 7. (Optional) Transcribe any other non-silent stems
extra_stems = ["drums", "bass", "vocals"]  # skip 'other' already done

client = Client(SPACE_ID)
for stem_name in extra_stems:
    stem_path = stems_dir / f"{stem_name}.wav"
    if not stem_path.exists():
        continue
    db = stem_loudness.get(stem_name, rms_db(stem_path))
    if db < silence_threshold_db:
        print(f"Skipping silent {stem_name}.wav ({db:.1f} dBFS)")
        continue
    print(f"Transcribing {stem_name}.wav ({db:.1f} dBFS)...")
    result = client.predict(
        audio_filepath=handle_file(str(stem_path)),
        api_name="/process_audio",
    )
    midi_bytes = extract_midi_from_html(result)
    out_path = out_dir / f"{song_name}_{stem_name}_yourmt3.mid"
    out_path.write_bytes(midi_bytes)
    print(f"Saved: {out_path} ({len(midi_bytes)} bytes)")
    files.download(str(out_path))